# Stage 02: Data Preprocessing, Feature Engineering & Dataset Preparation

**Project**: Project 18 — Water Quality Prediction via Gradient Boosting & Deep Neural Networks  
**Course**: Semester 5 Applied Machine Learning & Deep Learning (AML / DL)  
**Authoritative Module**: [`src/preprocessing.py`](../../src/preprocessing.py)  
**Specification**: [`docs/PREPROCESSING_SPECIFICATION.md`](../../docs/PREPROCESSING_SPECIFICATION.md)  
**Input Source**: `data/Dataset/Parquet/Combined Data/Combined_dataset.parquet` (4,297,895 rows × 14 cols)  
**Output Target Directory**: `data/GeneratedOutput/02.Preprocessing&FeatureSelection/`  

---

### Pipeline Objectives & Sequence
1. **Input Dataset & Schema Validation**: Ingest multi-country dataset, standardise schema, cast semantic data types.
2. **Deterministic Data Cleaning**: Neutralize sensor hardware sentinels (-999999, corrupt pH), convert USA Fahrenheit readings, clean sub-detection-limit negative concentrations, drop exact ingestion duplicates, and append non-destructive suspicious value flags.
3. **Domain-Grounded Feature Engineering**: Generate stoichiometry nutrient ratios ($N:P$), organic pollution ratios ($BOD:DO$), signed oxygen saturation deficit ($DO_{\text{deficit}}$), and circular seasonal cyclical encodings (`month_sin`, `month_cos`).
4. **Target & Input-Feature Segregation**: Segregate continuous regression target (`ccme_values`) and categorical/ordinal classification targets (`ccme_wqi`, `ccme_wqi_code`) from the 15 predictor features to prevent leakage.
5. **Leakage-Free Splitting Strategy**: Implement 70/15/15 stratified random split (benchmark) with spatial station-group and temporal forecasting evaluation options.
6. **Training-Only Imputation & Feature Scaling**: Fit group medians by `(country, waterbody_type)` and `(country, month)` on the 70% train split only; fit `StandardScaler` on train inputs only.
7. **Output Artifact Validation**: Verify zero missing values, zero infinite values, and identical schema across all partitions.
8. **Final Dataset Readiness Summary**: Audit locally generated Parquet datasets and metadata for Stage 03 Model Training.


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Ensure project root is on sys.path
for candidate in [".", "../..", ".."]:
    if os.path.exists(os.path.join(candidate, "src", "preprocessing.py")):
        PROJECT_ROOT = os.path.abspath(candidate)
        break

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import (
    WaterQualityPreprocessor,
    run_cleaning_pipeline,
    engineer_features,
    split_data,
    impute_missing_values,
    scale_features,
    prepare_training_datasets,
    BASE_NUMERIC_FEATURES,
    ENGINEERED_FEATURES,
    MODEL_INPUT_FEATURES,
    TARGET_REGRESSION,
    TARGET_CLASSIFICATION,
    TARGET_CLASSIFICATION_ORDINAL,
    CCME_WQI_CATEGORIES,
    CCME_WQI_MAP,
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Base Features ({len(BASE_NUMERIC_FEATURES)}): {BASE_NUMERIC_FEATURES}")
print(f"Engineered Features ({len(ENGINEERED_FEATURES)}): {ENGINEERED_FEATURES}")
print(f"Model Input Features ({len(MODEL_INPUT_FEATURES)}): {MODEL_INPUT_FEATURES}")


Project root : /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL
Base Features (8): ['ph', 'temperature_c', 'do_mg_l', 'bod_mg_l', 'ammonia_mg_l', 'orthophosphate_mg_l', 'nitrogen_mg_l', 'nitrate_mg_l']
Engineered Features (7): ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']
Model Input Features (15): ['ph', 'temperature_c', 'do_mg_l', 'bod_mg_l', 'ammonia_mg_l', 'orthophosphate_mg_l', 'nitrogen_mg_l', 'nitrate_mg_l', 'np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


## 1. Input Dataset & Schema Validation
Inspect the master dataset schema, column types, and sampling station coverage.


In [2]:
raw_path = os.path.join(PROJECT_ROOT, "data", "Dataset", "Parquet", "Combined Data", "Combined_dataset.parquet")
raw_schema = pq.read_schema(raw_path)

print("Master Parquet Dataset Schema:")
for name, typ in zip(raw_schema.names, raw_schema.types):
    print(f"  {name:<35} : {typ}")

# Quick summary profile
raw_table = pq.read_table(raw_path, columns=["Country", "Area", "Waterbody Type", "Date", "CCME_Values", "CCME_WQI"])
print(f"\nTotal Master Observations : {raw_table.num_rows:,} rows")
print(f"Unique Countries          : {len(np.unique(raw_table['Country'].to_numpy()))} countries")
print(f"Unique Sampling Stations  : {len(np.unique(raw_table['Area'].to_numpy())):,} stations")


Master Parquet Dataset Schema:
  Country                             : large_string
  Area                                : large_string
  Waterbody Type                      : large_string
  Date                                : large_string
  Ammonia (mg/l)                      : double
  Biochemical Oxygen Demand (mg/l)    : double
  Dissolved Oxygen (mg/l)             : double
  Orthophosphate (mg/l)               : double
  pH (ph units)                       : double
  Temperature (cel)                   : double
  Nitrogen (mg/l)                     : double
  Nitrate (mg/l)                      : double
  CCME_Values                         : double
  CCME_WQI                            : large_string



Total Master Observations : 4,297,895 rows


Unique Countries          : 7 countries


Unique Sampling Stations  : 64,665 stations


## 2. Deterministic Data Cleaning
We load the authoritative pre-cleaned dataset (or execute the cleaning pipeline via `run_cleaning_pipeline()`).  
The cleaning resolves telemetry errors (-999999), converts historical USA river Fahrenheit readings, applies column-specific negative concentration rules, removes exact duplicate records, and appends boolean quality audit flags.


In [3]:
cleaned_path = os.path.join(PROJECT_ROOT, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection", "cleaned_water_quality.parquet")

if os.path.exists(cleaned_path):
    print(f"Loading authoritative cleaned dataset from {cleaned_path}...")
    df_clean = pd.read_parquet(cleaned_path)
else:
    print("Running cleaning pipeline from raw source...")
    df_clean, summary = run_cleaning_pipeline(raw_path=raw_path, output_path=cleaned_path)

print(f"Cleaned dataset shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
df_clean[["country", "area", "waterbody_type", "date", "ph", "temperature_c", "do_mg_l", "ccme_values", "ccme_wqi"]].head(5)


Loading authoritative cleaned dataset from /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/data/GeneratedOutput/02.Preprocessing&FeatureSelection/cleaned_water_quality.parquet...


Cleaned dataset shape: 4,287,890 rows x 23 columns


,country,area,waterbody_type,date,ph,temperature_c,do_mg_l,ccme_values,ccme_wqi
0,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2014-12-01,NaN,24.7,4.15,93.102127,Good
1,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-02-02,NaN,22.4,12.07,100.000000,Excellent
2,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-03-02,NaN,17.0,10.59,100.000000,Excellent
3,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-04-13,7.4,17.0,3.42,100.000000,Excellent
4,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-01-15,7.4,23.2,3.91,92.814413,Good


## 3. Geochemical & Seasonal Feature Engineering
We generate domain-grounded environmental features:
- **$N:P$ Nutrient Ratio** (`np_ratio`): Stoichiometric ratio of total nitrogen to orthophosphate (Redfield eutrophication threshold = 16:1). Safe division produces `NaN` when phosphorus is zero or negative.
- **$BOD:DO$ Ratio** (`bod_do_ratio`): Organic pollutant oxygen demand relative to available dissolved oxygen. Safe division handles zero/anoxic DO.
- **Signed Dissolved Oxygen Deficit** (`do_deficit`): $DO_{\text{sat}}(T) - DO$. Positive deficit captures microbial oxygen depletion from sewage/organic runoff; negative deficit preserves photosynthetic algal supersaturation events.
- **Cyclical Month Encoding** (`month_sin`, `month_cos`): Circular angle $(month - 1)/12$ ensuring January ($0^\circ$) and December ($330^\circ$) are adjacent in circular feature space.


In [4]:
df_feat, new_cols = engineer_features(df_clean)
print(f"Engineered features added: {new_cols}")

# Display engineered features distribution
df_feat[["np_ratio", "bod_do_ratio", "do_deficit", "month_sin", "month_cos", "ccme_wqi_code"]].describe().round(4)


2026-10-23 02:35:49,287 [INFO] Engineered 7 new features: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


Engineered features added: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


,np_ratio,bod_do_ratio,do_deficit,month_sin,month_cos,ccme_wqi_code
count,2.817393e+06,2.821499e+06,3.140827e+06,4.287890e+06,4.287890e+06,4287890.0
mean,6.016730e+01,1.918900e+01,9.480000e-01,-3.500000e-03,-3.000000e-02,3.1843
std,3.630478e+02,3.953171e+03,1.941900e+00,7.105000e-01,7.031000e-01,1.0555
min,0.000000e+00,0.000000e+00,-1.691930e+01,-1.000000e+00,-1.000000e+00,0.0
25%,1.000000e+01,1.569000e-01,-3.750000e-02,-8.660000e-01,-8.660000e-01,3.0
50%,2.613640e+01,2.348000e-01,6.833000e-01,0.000000e+00,-0.000000e+00,4.0
75%,4.980390e+01,3.167000e-01,1.528600e+00,8.660000e-01,5.000000e-01,4.0
max,1.490000e+05,4.210000e+06,1.465200e+01,1.000000e+00,1.000000e+00,4.0


## 4. Target & Input-Feature Segregation
To prevent data leakage, target variables and target-derived flags are strictly isolated from the predictor matrix:
- **Input Feature Matrix ($X$)**: 15 clean predictors (`MODEL_INPUT_FEATURES`).
- **Continuous Regression Target ($y_{\text{reg}}$)**: `ccme_values` (0.0 to 100.0).
- **Categorical / Ordinal Target ($y_{\text{clf}}$)**: `ccme_wqi` / `ccme_wqi_code` (0: Poor -> 4: Excellent).


In [5]:
print("Modeling Features List (15 predictors):")
for idx, feat in enumerate(MODEL_INPUT_FEATURES, 1):
    print(f"  {idx:>2}. {feat}")

# Verify no targets exist in feature set
for target in [TARGET_REGRESSION, TARGET_CLASSIFICATION, TARGET_CLASSIFICATION_ORDINAL]:
    assert target not in MODEL_INPUT_FEATURES, f"CRITICAL LEAKAGE: Target {target} is inside input features!"
print("\nLeakage check PASSED: Target columns strictly excluded from input feature list.")


Modeling Features List (15 predictors):
   1. ph
   2. temperature_c
   3. do_mg_l
   4. bod_mg_l
   5. ammonia_mg_l
   6. orthophosphate_mg_l
   7. nitrogen_mg_l
   8. nitrate_mg_l
   9. np_ratio
  10. bod_do_ratio
  11. do_deficit
  12. month
  13. quarter
  14. month_sin
  15. month_cos

Leakage check PASSED: Target columns strictly excluded from input feature list.


## 5. Train / Validation / Test Splitting Strategy
We implement a reproducible 70% Train / 15% Validation / 15% Test partition stratified by `ccme_wqi` (benchmark).  
In addition, `split_data()` supports `country_area_group` (disjoint geographic area/catchment clusters, noting that station-level isolation cannot be claimed due to heterogeneous area granularity) and `temporal_holdout` (future-year forecasting).


In [6]:
train_raw, val_raw, test_raw = split_data(
    df_feat,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    method="stratified_random",
    random_state=42,
)

print(f"Train split : {len(train_raw):,} rows ({len(train_raw)/len(df_feat)*100:.1f}%)")
print(f"Val split   : {len(val_raw):,} rows ({len(val_raw)/len(df_feat)*100:.1f}%)")
print(f"Test split  : {len(test_raw):,} rows ({len(test_raw)/len(df_feat)*100:.1f}%)")

# Verify class representation across partitions
class_summary = pd.DataFrame({
    "Train %": train_raw["ccme_wqi"].value_counts(normalize=True) * 100,
    "Val %": val_raw["ccme_wqi"].value_counts(normalize=True) * 100,
    "Test %": test_raw["ccme_wqi"].value_counts(normalize=True) * 100,
})
class_summary.round(2)


2026-10-23 02:36:07,823 [INFO] Split data (stratified_random): Train=3,001,523 (70.0%), Val=643,183 (15.0%), Test=643,184 (15.0%)


Train split : 3,001,523 rows (70.0%)
Val split   : 643,183 rows (15.0%)
Test split  : 643,184 rows (15.0%)


,Train %,Val %,Test %
ccme_wqi,,,
Excellent,50.06,50.06,50.06
Good,31.89,31.89,31.89
Marginal,7.67,7.67,7.67
Fair,7.42,7.42,7.42
Poor,2.96,2.96,2.96


## 6. Training-Only Imputation & Feature Scaling
To prevent test leakage:
1. **Group Median Imputation**: Group medians by `(country, waterbody_type)` and seasonal medians by `(country, month)` are learned **solely on `train_raw`** and applied to validation and test sets with hierarchical fallback.
2. **Standard Scaling**: `StandardScaler` is fitted **solely on `train_imp[MODEL_INPUT_FEATURES]`** for Deep Learning (MLP). Tree models ingest unscaled features. Targets are never scaled.


In [7]:
# 1. Impute missing values using train-only fitted statistics
train_imp, val_imp, test_imp, imputer_info = impute_missing_values(
    train_raw, val_raw, test_raw, feature_cols=MODEL_INPUT_FEATURES
)

# 2. Fit StandardScaler on train inputs only
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_imp, val_imp, test_imp, feature_cols=MODEL_INPUT_FEATURES
)

print("Remaining Missing Values in Modeling Features:")
print(f"  Train : {train_imp[MODEL_INPUT_FEATURES].isna().sum().sum()}")
print(f"  Val   : {val_imp[MODEL_INPUT_FEATURES].isna().sum().sum()}")
print(f"  Test  : {test_imp[MODEL_INPUT_FEATURES].isna().sum().sum()}")

# Scaler verification
train_means = train_scaled[MODEL_INPUT_FEATURES].mean().abs()
train_stds = train_scaled[MODEL_INPUT_FEATURES].std(ddof=0)
print(f"\nScaler Mean Check (max absolute deviation from 0.0): {train_means.max():.6e}")
print(f"Scaler Std Check  (max deviation from 1.0)         : {abs(train_stds - 1.0).max():.6e}")


Remaining Missing Values in Modeling Features:
  Train : 0
  Val   : 0
  Test  : 0



Scaler Mean Check (max absolute deviation from 0.0): 1.316204e-15
Scaler Std Check  (max deviation from 1.0)         : 2.220446e-16


## 7. Output Validation
Verify that all generated partition schemas, feature orders, and data types are aligned.


In [8]:
# Verify column order and integrity across partitions
assert list(train_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(val_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(test_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(train_scaled[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES

# Verify target integrity
assert train_imp["ccme_values"].notna().all()
assert val_imp["ccme_values"].notna().all()
assert test_imp["ccme_values"].notna().all()
assert train_imp["ccme_wqi_code"].notna().all()

print("Output Validation PASSED:")
print("  - Feature columns maintain identical 15-variable order across all partitions.")
print("  - Zero missing values or infinities exist in model inputs.")
print("  - Target columns are complete and valid.")


Output Validation PASSED:
  - Feature columns maintain identical 15-variable order across all partitions.
  - Zero missing values or infinities exist in model inputs.
  - Target columns are complete and valid.


## 8. Dataset Statistics & Final Readiness Summary
Summary of all locally generated Parquet datasets and provenance metadata in `data/GeneratedOutput/02.Preprocessing&FeatureSelection/`.


In [9]:
meta_path = os.path.join(PROJECT_ROOT, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection", "preprocessing_metadata.json")
with open(meta_path) as f:
    meta = json.load(f)

print("=" * 65)
print("STAGE 02 PREPROCESSING & FEATURE SELECTION: READINESS SUMMARY")
print("=" * 65)
print(f"Metadata Version      : {meta['metadata_version']}")
print(f"Total Master Records  : {meta['original_row_count']:,}")
print(f"Cleaned Records       : {meta['cleaned_row_count']:,}")
print(f"Exact Duplicates Drop : {meta['exact_duplicates_removed']:,}")
print(f"Fahrenheit Converted  : {meta['usa_temperature_degf_converted']:,} readings")
print(f"Sentinels Neutralized : {meta['sentinels_neutralized']} records")
print(f"Train Split Rows      : {meta['split_counts']['train']:,} (70.0%)")
print(f"Val Split Rows        : {meta['split_counts']['val']:,} (15.0%)")
print(f"Test Split Rows       : {meta['split_counts']['test']:,} (15.0%)")
print(f"Predictor Features    : {meta['input_feature_count']} features")
print(f"Target Classes        : {meta['target_class_order']}")
print("-" * 65)
print("Local Parquet Artifacts:")
for name, rel_path in meta["output_artifacts"].items():
    full_path = os.path.join(PROJECT_ROOT, rel_path)
    size_mb = os.path.getsize(full_path) / (1024 * 1024) if os.path.exists(full_path) else 0.0
    print(f"  {name:<16} : {rel_path} ({size_mb:.2f} MB)")
print("=" * 65)
print("\nVERDICT: READY FOR STAGE 03 MODEL TRAINING (Tree Baselines & Deep MLP)")


STAGE 02 PREPROCESSING & FEATURE SELECTION: READINESS SUMMARY
Metadata Version      : 2.0.0
Total Master Records  : 4,297,895
Cleaned Records       : 4,287,890
Exact Duplicates Drop : 10,005
Fahrenheit Converted  : 5,852 readings
Sentinels Neutralized : 84 records
Train Split Rows      : 3,001,523 (70.0%)
Val Split Rows        : 643,183 (15.0%)
Test Split Rows       : 643,184 (15.0%)
Predictor Features    : 15 features
Target Classes        : ['Poor', 'Marginal', 'Fair', 'Good', 'Excellent']
-----------------------------------------------------------------
Local Parquet Artifacts:
  cleaned_data     : data/GeneratedOutput/02.Preprocessing&FeatureSelection/cleaned_water_quality.parquet (60.94 MB)
  train_unscaled   : data/GeneratedOutput/02.Preprocessing&FeatureSelection/train.parquet (115.34 MB)
  val_unscaled     : data/GeneratedOutput/02.Preprocessing&FeatureSelection/val.parquet (23.35 MB)
  test_unscaled    : data/GeneratedOutput/02.Preprocessing&FeatureSelection/test.parquet (23.3